In [0]:
select * from occupazione
limit 3;

-- Find average employment (occupazione) per country. 
select country, round(avg(obs_value),2) as aavg_emp_rate
from occupazione
group by country 
order by country asc;

--	Find max unemployment per year. 
select year, max(obs_value) as max_unemp_rate
from disoccupazione
group by year
order by year asc;

-- 	Group by sex and calculate average obs_value.
select sex,round(avg(obs_value),2) as avg_sex_rate
from disoccupazione
group by sex
order by sex asc;

-- 	Find countries where unemployment > 10. 
select distinct country
from disoccupazione
where obs_value > 10
order by country asc;

-- Join both tables on: iso_code, country, sex, age, year
select d.iso_code, o.country, o.sex, d.age, d.year,
      d.obs_value as disoccupazione, o.obs_value as occupazione
from disoccupazione d
join occupazione o
on d.iso_code = o.iso_code and d.country = o.country
and d.sex = o.sex and d.age = o.age and d.year = o.year;

/* 13.	Find countries where unemployment > employment */
select o.country, o.sex, d.age, d.year,
      d.obs_value as unemployment, o.obs_value as employment
from disoccupazione d
join occupazione o
on d.iso_code = o.iso_code and d.country = o.country
and d.sex = o.sex and d.age = o.age and d.year = o.year
where d.obs_value > o.obs_value
order by o.country asc;


-- Compute difference: employment - unemployment
select o.country, o.sex, d.age, d.year,
      d.obs_value as unemployment, o.obs_value as employment,
      round(o.obs_value - d.obs_value, 2) as gap
from disoccupazione d
join occupazione o
on d.iso_code = o.iso_code and d.country = o.country
and d.sex = o.sex and d.age = o.age and d.year = o.year
order by o.year,o.country asc;

-- Find top 5 countries with highest unemployment gap. 
with cal as (
  select o.country,d.year,
        sum(d.obs_value) as unemployment, sum(o.obs_value) as employment,
        round(sum(o.obs_value) - sum(d.obs_value), 2) as gap
  from disoccupazione d
  join occupazione o
  on d.iso_code = o.iso_code and d.country = o.country
  and d.sex = o.sex and d.age = o.age and d.year = o.year
  group by o.country,d.year )
select *
from cal
where gap < 0
order by gap asc
limit 5;


-- Use window function to rank countries by unemployment per year. 
with rnk as (
  select country, year,
        round(sum(obs_value),2) as unemployment
  from disoccupazione
  group by country, year
)
select *, rank() over(partition by year order by unemployment desc) as rnk
from rnk;


-- # Calculate employment ratio:
with cal as (
  select o.country,d.year,
        round(sum(d.obs_value),2) as unemployment, 
        round(sum(o.obs_value),2) as employment,
        round(sum(o.obs_value) + sum(d.obs_value), 2) as parti
  from disoccupazione d
  join occupazione o
  on d.iso_code = o.iso_code and d.country = o.country
  and d.sex = o.sex and d.age = o.age and d.year = o.year
  group by o.country,d.year )
select country, year, employment, unemployment,
      round((employment/parti),2) as ratio
from cal
order by country, year;


-- Find unemployment percentage per country
with cal as (
  select o.country,d.year,
        round(sum(d.obs_value),2) as unemployment, 
        round(sum(o.obs_value),2) as employment,
        round(sum(o.obs_value) + sum(d.obs_value), 2) as parti
  from disoccupazione d
  join occupazione o
  on d.iso_code = o.iso_code and d.country = o.country
  and d.sex = o.sex and d.age = o.age and d.year = o.year
  group by o.country,d.year )
select country, year, employment, unemployment,
      round((unemployment * 100/parti),2) as ratio
from cal
order by country, year;

/*•	Label countries as: 
	"High Unemployment" if obs_value > threshold 
	"Low Unemployment" otherwise 
*/
with avg_val as (
  select avg(obs_value) as avg_unemp 
  from disoccupazione
)
select d.country, d.year,
      case 
        when d.obs_value > a.avg_unemp then 'High Unemployment'
        else 'Low Unemployment'
      end as label
from disoccupazione d
cross join avg_val a
order by d.country, d.year;

-- Calculate year-over-year change in obs_value. 
with tab as (
  select country, year,
        round(sum(obs_value),2) as unemployment
  from disoccupazione
  group by country, year
),
final as (
  select *,
        lag(unemployment) over(partition by country order by year) as prev_unemp
  from tab
)
select *,
      round(unemployment - prev_unemp,2) as change
from final;


-- Find growth rate of employment year over year
with tab as (
  select country, year,
        round(sum(obs_value),2) as employment
  from occupazione
  group by country, year
),
final as (
  select *,
        lag(employment) over(partition by country order by year) as prev_emp
  from tab
)
select *,
      round(
            (employment - prev_emp) * 100 / nullif(prev_emp, 0),
            2) as growth_rate
from final;

-- Identify years where unemployment increased compared to previous year.  
with tab as (
  select country, year,
        round(sum(obs_value),2) as unemployment
  from disoccupazione
  group by country, year
),
final as (
  select *,
        lag(unemployment) over(partition by country order by year) as prev_unemp
  from tab
)
select *
from final
where unemployment > prev_unemp
order by country, year;

-- Pivot sex column → male vs female comparison. 
with total as (
      select o.country, o.sex, d.year,
      d.obs_value as unemployment, o.obs_value as employment,
      round(o.obs_value - d.obs_value, 2) as gap
      from disoccupazione d
      join occupazione o
      on d.iso_code = o.iso_code and d.country = o.country
      and d.sex = o.sex and d.age = o.age and d.year = o.year
)
select country, year, 
      round(sum(case when sex = 'Male' then employment else 0 end),2) as male_emp,
      round(sum(case when sex = 'Female' then employment else 0 end),2) as female_emp,
      round(sum(case when sex = 'Male' then unemployment else 0 end),2) as male_Unemp,
      round(sum(case when sex = 'Female' then unemployment else 0 end),2) as female_Unemp
from total
group by country, year
order by country, year;


